In [37]:
import torch
import torch.nn.functional as F

In [38]:
chemicals = open('/Users/raghav/Downloads/chemicals.txt', 'r').read().splitlines()

In [39]:
chemicals = [w for w in chemicals if w.isalpha() and all(ord(c) < 128 for c in w)]

In [40]:
len(chemicals)

1727

In [41]:
chemicals[:5]

['acanthite', 'acetaldehyde', 'acetamide', 'acetate', 'acetic']

In [42]:
chars = sorted(list(set(''.join(chemicals))))
chars

['a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z']

In [43]:
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
vocab_size = len(stoi)
stoi

{'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26,
 '.': 0}

In [44]:
itos = {i:s for s,i in stoi.items()}
itos

{1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z',
 0: '.'}

In [45]:
vocab_size

27

In [46]:
block_size = 3

def build_dataset(chemicals):
    X , Y = [] , []
    for chemical in chemicals:
        context = [0] * block_size
        for ch in chemical + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)

    return X,Y

import random
random.seed(42)
random.shuffle(chemicals)
n1 = int(0.8*len(chemicals))
n2 = int(0.9*len(chemicals))

Xtr ,Ytr = build_dataset(chemicals[:n1])
Xdev , Ydev = build_dataset(chemicals[n1:n2])
Xte , Yte = build_dataset(chemicals[n2:])

In [48]:
Xtr.shape , Ytr.shape

(torch.Size([14071, 3]), torch.Size([14071]))

In [51]:
class Linear:
    def __init__(self,fan_in,fan_out,bias=True):
        self.weight = torch.randn((fan_in,fan_out),generator = g)
        self.bias = torch.randn(fan_out) if bias else None

    def __call__(self,x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out

    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])

class BatchNorm1d:
    def __init__(self,dim,eps=1e-5,momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True

        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self,x):
        if self.training:
            xmean = x.mean(0,keepdim = True)
            xvar = x.var(0,keepdim = True)
        else:
            xmean = self.running_mean
            xvar = self.running_var

        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta

        if self.training:
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar

        return self.out

    def parameters(self):
        return [self.gamma,self.beta]

class Tanh:
    def __call__(self,x):
        self.out = torch.tanh(x)
        return self.out
    def parameters(self):
        return []


In [55]:
g = torch.Generator().manual_seed(2147483647)

n_embd = 10
n_hidden = 100

C = torch.randn(vocab_size,n_embd,generator = g)

layers = [
    Linear((n_embd*block_size),n_hidden,bias=False),BatchNorm1d(n_hidden),Tanh(),
    Linear(n_hidden,n_hidden,bias=False),BatchNorm1d(n_hidden),Tanh(),
    Linear(n_hidden,n_hidden,bias=False),BatchNorm1d(n_hidden),Tanh(),
    Linear(n_hidden,n_hidden,bias=False),BatchNorm1d(n_hidden),Tanh(),
    Linear(n_hidden,n_hidden,bias=False),BatchNorm1d(n_hidden),Tanh(),
    Linear(n_hidden,vocab_size,bias=False),BatchNorm1d(vocab_size),
]

parameters = [C] + [p for layer in layers for p in layer.parameters()]
print(sum(p.nelement() for p in parameters)) 

47024


In [56]:
for p in parameters:
    p.requires_grad = True

In [57]:
max_steps = 200000
batch_size = 32
lossi = []

for i in range(max_steps):

    ix = torch.randint(0,Xtr.shape[0],(batch_size,))
    Xb, Yb = Xtr[ix] , Ytr[ix]

    embd = C[Xb]
    x = embd.view(embd.shape[0],-1)

    for layer in layers:
        x = layer(x)

    loss = F.cross_entropy(x,Yb)

    for p in parameters:
        p.grad = None

    loss.backward()

    lr = 0.1 if i < 150000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    if i % 10000 == 0: 
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')

    lossi.append(loss.log10().item())

      0/ 200000: 3.6346
  10000/ 200000: 1.3335
  20000/ 200000: 1.0175
  30000/ 200000: 1.4196
  40000/ 200000: 1.2358
  50000/ 200000: 1.1423
  60000/ 200000: 0.9827
  70000/ 200000: 0.8608
  80000/ 200000: 1.4041
  90000/ 200000: 0.9548
 100000/ 200000: 1.2299
 110000/ 200000: 0.8808
 120000/ 200000: 1.2201
 130000/ 200000: 0.9271
 140000/ 200000: 1.0982
 150000/ 200000: 0.8652
 160000/ 200000: 1.2298
 170000/ 200000: 1.5923
 180000/ 200000: 1.2259
 190000/ 200000: 1.4095


In [59]:
for layer in layers:
    layer.training = False



with torch.no_grad():
    embd = C[Xtr]
    x = embd.view(embd.shape[0],-1)
    
    for layer in layers:
        x = layer(x)
    
    loss = F.cross_entropy(x,Ytr)

print("train :", loss.item())

train : tensor(1.0546)


In [61]:
for layer in layers:
    layer.training = False

with torch.no_grad():
    embd = C[Xdev]
    x = embd.view(embd.shape[0],-1)
    
    for layer in layers:
        x = layer(x)
    
    loss = F.cross_entropy(x,Ydev)

print("dev :", loss.item())

dev : 1.5171421766281128


In [62]:

for _ in range(20):
    out = []
    context = [0] * block_size
    while True:

        emb = C[torch.tensor([context])]
        x = emb.view(emb.shape[0],-1)

        for layer in layers:
            x = layer(x)
            
        logits = x
        probs = F.softmax(logits,dim=1)
        ix = torch.multinomial(probs,num_samples=1,generator = g).item()

        context = context[1:] + [ix]
        out.append(ix)

        if ix == 0:
            break

    print(''.join(itos[i] for i in out))

glycol.
glyceronitrol.
amine.
nite.
iridinin.
silicassium.
butantadecamine.
salite.
octylene.
sillite.
pyric.
iride.
octadecamine.
tungstium.
.
albate.
andecoxide.
tethorgane.
heptadecsulfone.
none.
